In [ ]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_curve
import numpy as np
import torch.nn as nn
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

In [ ]:
normal = pd.read_csv("../normal.csv")
attack = pd.read_csv("../attack.csv")
merged = pd.concat([normal, attack], ignore_index=True)

In [ ]:
def complete_swat_preprocessing(merged):
    """
    Complete preprocessing pipeline specifically for SWaT dataset
    """
    print(f"Initial shape: {merged.shape}")
    print(f"Columns: {merged.columns.tolist()}")
    
    df_proc = merged.copy()
    
    # Fix column names - strip whitespace
    df_proc.columns = df_proc.columns.str.strip()
    
    # Step 2: Remove duplicates (safe for SWaT)
    initial_len = len(df_proc)
    df_proc = df_proc.drop_duplicates()
    print(f"Removed {initial_len - len(df_proc)} duplicate rows")
    
    # Step 3: Separate features and target
    target = df_proc['attack']
    features = df_proc.drop(columns=['attack','Timestamp'])
    
    # Step 4: Fix column names in features (strip whitespace)
    features.columns = features.columns.str.strip()
    
    # Step 5: Handle missing values based on SWaT domain knowledge
    # Binary actuators (fill with 0 for OFF state)
    binary_actuators = ['MV101', 'MV201', 'P201', 'P202', 'P204', 'MV303']
    for col in binary_actuators:
        matching_cols = [c for c in features.columns if c.strip() == col]
        if matching_cols:
            actual_col = matching_cols[0]
            features[actual_col] = features[actual_col].fillna(0)
            features[actual_col] = features[actual_col].astype(int)
            if actual_col != col:
                features = features.rename(columns={actual_col: col})
    
    # Critical continuous sensors (interpolate)
    critical_sensors = ['LIT101', 'AIT201', 'AIT202', 'FIT401', 'PIT501']
    for col in critical_sensors:
        matching_cols = [c for c in features.columns if c.strip() == col]
        if matching_cols:
            actual_col = matching_cols[0]
            features[actual_col] = features[actual_col].interpolate(method='linear', limit_direction='both') #Linear interpolation fills missing values based on surrounding values.
            if actual_col != col:
                features = features.rename(columns={actual_col: col})
    
    # Other sensors (fill with forward fill then median)
    numeric_cols = features.select_dtypes(include=[np.number]).columns
    other_sensors = [col for col in numeric_cols if col not in binary_actuators + critical_sensors]
    for col in other_sensors:
        features[col] = features[col].ffill().fillna(features[col].median())
    
    
    # Step 7: Encode target
    if target.dtype == 'object':
        target_encoded = (target.str.strip() == 'Attack').astype(int)
    else:
        target_encoded = target
    
    print(f"\nFinal shape: {features.shape}")
    print(f"Missing values: {features.isnull().sum().sum()}")
    print(f"Attack samples: {target_encoded.sum()}")
    print(f"Normal samples: {len(target_encoded) - target_encoded.sum()}")
    
    return features, target_encoded

In [ ]:
X_swat, y_swat = complete_swat_preprocessing(merged)


print(f" Features shape: {X_swat.shape}")
print(f" Target shape: {y_swat.shape}")
print(f" Attack samples: {y_swat.sum()} ({y_swat.mean()*100:.2f}%)")
print(f" Normal samples: {len(y_swat) - y_swat.sum()} ({(1-y_swat.mean())*100:.2f}%)")
print(f" Missing values in features: {X_swat.isnull().sum().sum()}")

In [ ]:
#finding attack indecies
attack_indices = y_swat[y_swat == 1].index

print("First attack:", attack_indices.min())
print("Last attack:", attack_indices.max())
print("Total attacks:", len(attack_indices))

In [ ]:
split_index = int(len(X_swat) * 0.7)

X_train = X_swat.iloc[:split_index]
y_train = y_swat.iloc[:split_index]

X_test = X_swat.iloc[split_index:]
y_test = y_swat.iloc[split_index:]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=False)

In [ ]:
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        self.mu = nn.Linear(32, latent_dim)
        self.logvar = nn.Linear(32, latent_dim)
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        mu = self.mu(h)
        logvar = self.logvar(h)
        
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        
        return self.decoder(z), mu, logvar

In [ ]:
def vae_loss(x, recon, mu, logvar, beta=0.02):
    recon_loss = torch.mean((x - recon) ** 2)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VAE(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        x = batch[0].to(device)
        
        optimizer.zero_grad()
        recon, mu, logvar = model(x)
        
        loss = vae_loss(x, recon, mu, logvar)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.6f}")

Why  huge

Because:

using mean squared error over 51 features

aggregated → values become large

👉 Totally fine — threshold is relative, not absolute

In [ ]:
model.eval()

with torch.no_grad():
    recon_test, _, _ = model(X_test_tensor.to(device))
    
recon_errors = torch.mean((X_test_tensor.to(device) - recon_test)**2, dim=1).cpu().numpy()

In [ ]:
from sklearn.metrics import f1_score

best_f1 = 0
best_threshold = 0

for t in np.linspace(recon_errors.min(), recon_errors.max(), 200):
    y_pred = (recon_errors > t).astype(int)
    
    f1 = f1_score(y_test, y_pred)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

In [ ]:
# Reduce threshold by ~5–10%
threshold = best_threshold * 0.9
window = 10
recon_errors_smooth = np.convolve(recon_errors, np.ones(window)/window, 'same')
y_pred = (recon_errors_smooth > threshold).astype(int)

In [ ]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
roc = roc_auc_score(y_test, recon_errors)
print("ROC AUC:", roc)

In [ ]:
import matplotlib.pyplot as plt

# Smooth reconstruction error
window = 10
recon_errors_smooth = np.convolve(recon_errors, np.ones(window)/window, 'same')

# Plot
plt.figure(figsize=(14,5))
plt.plot(recon_errors_smooth, label="Smoothed Reconstruction Error", color='blue')
plt.axhline(y=threshold, color='red', linestyle='--', label="Threshold")
plt.title("VAE Reconstruction Error and Threshold")
plt.xlabel("Sample Index")
plt.ylabel("Error")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch

print("\n--- Feature Importance for Anomalies (VAE) ---")
anomalies_idx = np.where(y_pred == 1)[0]
if len(anomalies_idx) > 0:
    X_anom = X_test_tensor.to(device)[anomalies_idx]
    recon_anom = recon_test[anomalies_idx]
    
    feature_impact = torch.mean((X_anom - recon_anom)**2, dim=0).cpu().numpy()
    
    fnames = features.columns if 'features' in locals() else [f'Feature_{i}' for i in range(len(feature_impact))]
    
    importance_df = pd.DataFrame({'Feature': fnames, 'MSE_Impact': feature_impact})
    importance_df = importance_df.sort_values(by='MSE_Impact', ascending=False)
    print(importance_df.head(40).to_string(index=False))
else:
    print("No anomalies detected.")

In [ ]:
# --- Test ONLY on attack.csv ---
print("Evaluating VAE ONLY on attack.csv...")
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

atk_df = pd.read_csv("../attack.csv")
label_col = 'attack'
feature_cols = [c for c in atk_df.columns if c != label_col and c != 'Timestamp' and c != 'label']
X_atk = atk_df[feature_cols].select_dtypes(include=[np.number]).values
X_atk = np.nan_to_num(X_atk, nan=0.0)

# The scaler is already trained as 'scaler'
X_atk_scaled = scaler.transform(X_atk)

y_raw_atk = atk_df[label_col].values
if y_raw_atk.dtype == object or isinstance(y_raw_atk[0], str):
    y_atk = np.where(y_raw_atk == 'Normal', 0, 1).astype(int)
else:
    y_atk = (y_raw_atk != 0).astype(int)

X_atk_tensor = torch.tensor(X_atk_scaled, dtype=torch.float32)

model.eval()
with torch.no_grad():
    recon_atk, _, _ = model(X_atk_tensor.to(device))

recon_errors_atk = torch.mean((X_atk_tensor.to(device) - recon_atk)**2, dim=1).cpu().numpy()

# threshold and window are already defined in the notebook
recon_errors_smooth_atk = np.convolve(recon_errors_atk, np.ones(window)/window, 'same')
y_pred_atk = (recon_errors_smooth_atk > threshold).astype(int)

print("Classification Report on attack.csv:")
print(classification_report(y_atk, y_pred_atk))
print("Confusion Matrix on attack.csv:")
print(confusion_matrix(y_atk, y_pred_atk))
